# Knowledge Distillation on ImageNet-1k — ResNet50 → MobileNetV2

This project studies **knowledge distillation (KD)** on **ImageNet-1k**, distilling
a strong **ResNet50** *teacher* into a lightweight **MobileNetV2** *student*.

**Goal.** Compare a MobileNetV2 student trained with a standard, high-accuracy
ImageNet recipe (no overfitting) against the **same** student trained with three
distillation strategies. The student recipe (optimizer, LR, schedule, epochs,
augmentation) is held **fixed**; only the **distillation method** changes, so the
comparison is fair.

| ID | Model | Objective |
|----|-------|-----------|
| **E1** | ResNet50 (teacher) | pretrained ImageNet weights (reference accuracy) |
| **E2** | MobileNetV2 | Baseline — CrossEntropy only |
| **E3** | MobileNetV2 | **Response-based KD** (Hinton soft logits) |
| **E4** | MobileNetV2 | **Attention Transfer (AT)** |
| **E5** | MobileNetV2 | **Response KD + Attention Transfer** |

**Teacher.** ResNet50 is used with its pretrained `IMAGENET1K_V2` weights (~80%
top-1) — it is already trained on ImageNet, so we simply **freeze** it and use it
as a fixed knowledge source (no teacher fine-tuning needed).

> **Note on scale.** Full ImageNet-1k training is heavy (1.28M images, many
> GPU-hours per epoch). The notebook defaults to `CFG.quick_run = True` (a small
> subset + few epochs) so it runs end-to-end anywhere. Set `CFG.quick_run = False`
> and point `CFG.data_root` at a real ImageNet folder to run the full recipe.


## 1. Imports and Configuration

In [ ]:
import os
import time
import math
import random
from dataclasses import dataclass, field

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

import torchvision
from torchvision import transforms, models, datasets

import matplotlib.pyplot as plt

print("PyTorch     :", torch.__version__)
print("Torchvision :", torchvision.__version__)


In [ ]:
# ----- Device + mixed precision -----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
print("Device:", device, "| AMP:", USE_AMP)


@dataclass
class Config:
    # Point this at an ImageNet-1k folder with train/ and val/ subdirectories
    # (standard ImageFolder layout: data_root/train/<wnid>/*.JPEG, data_root/val/...).
    data_root: str = "/path/to/imagenet"
    image_size: int = 224
    num_classes: int = 1000
    num_workers: int = 8
    seed: int = 42
    ckpt_dir: str = "checkpoints_imagenet"

    # Train the student from scratch (proper KD study). Set True to instead
    # fine-tune a pretrained MobileNetV2.
    student_pretrained: bool = False

    # quick_run = small subset + few epochs so the notebook runs anywhere.
    # Set False to use the full standard recipe on the full dataset.
    quick_run: bool = True


CFG = Config()
os.makedirs(CFG.ckpt_dir, exist_ok=True)

# ImageNet normalisation statistics.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


# ----- Shared student recipe (high-accuracy ImageNet MobileNetV2 settings) -----
# These same values are reused for the baseline AND every KD experiment, so only
# the distillation method differs between runs.
RECIPE = {
    "epochs": 100,
    "batch_size": 256,
    "lr": 0.05,            # SGD base LR for batch 256
    "momentum": 0.9,
    "nesterov": True,
    "weight_decay": 4e-5,  # small WD is standard for MobileNetV2 (no overfitting)
    "label_smoothing": 0.1,
    "warmup_epochs": 5,
}

# ----- Distillation hyperparameters (the part we adjust per experiment) -----
KD = {
    "temperature": 4.0,    # softens teacher/student logits
    "kd_alpha": 0.9,       # weight on KD term; CE weight = 1 - kd_alpha
    "at_weight": 1000.0,   # beta for attention-transfer loss
}

# quick_run overrides so the whole pipeline executes in minutes on a small subset.
QUICK = {
    "epochs": 2,
    "batch_size": 64,
    "train_subset": 4000,   # number of training images to sample
    "val_subset": 2000,     # number of val images to sample
    "max_eval_batches": None,
}

print("quick_run:", CFG.quick_run)
CFG


## 2. Data — ImageNet-1k

Standard ImageNet transforms: `RandomResizedCrop(224)` + horizontal flip for
training; `Resize(256)` + `CenterCrop(224)` for evaluation. ImageNet is large
enough that this light augmentation reaches high accuracy **without overfitting**,
which is exactly why we keep it fixed across all experiments.


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(CFG.image_size),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(CFG.image_size),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def _maybe_subset(dataset, k, seed=CFG.seed):
    """Deterministically sample k items (for quick_run); k=None -> full dataset."""
    if k is None or k >= len(dataset):
        return dataset
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(dataset), generator=g)[:k].tolist()
    return Subset(dataset, idx)


def build_dataloaders():
    train_dir = os.path.join(CFG.data_root, "train")
    val_dir = os.path.join(CFG.data_root, "val")
    if not (os.path.isdir(train_dir) and os.path.isdir(val_dir)):
        raise FileNotFoundError(
            f"Expected ImageNet at {CFG.data_root} with train/ and val/ subfolders. "
            "Set CFG.data_root to your ImageNet path (ImageFolder layout).")

    bs = QUICK["batch_size"] if CFG.quick_run else RECIPE["batch_size"]

    train_ds = datasets.ImageFolder(train_dir, transform=train_transform)
    val_ds = datasets.ImageFolder(val_dir, transform=eval_transform)

    if CFG.quick_run:
        train_ds = _maybe_subset(train_ds, QUICK["train_subset"])
        val_ds = _maybe_subset(val_ds, QUICK["val_subset"])

    train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True,
                              num_workers=CFG.num_workers, pin_memory=USE_AMP,
                              drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False,
                            num_workers=CFG.num_workers, pin_memory=USE_AMP)
    print(f"train images: {len(train_ds):,} | val images: {len(val_ds):,} | batch: {bs}")
    return train_loader, val_loader


train_loader, val_loader = build_dataloaders()


## 3. Utilities — metrics, evaluation, checkpoints

In [ ]:
def set_seed(seed=CFG.seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def topk_correct(output, target, ks=(1, 5)):
    """Return {k: #correct in top-k} for this batch."""
    maxk = max(ks)
    _, pred = output.topk(maxk, dim=1, largest=True, sorted=True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))
    return {k: correct[:k].reshape(-1).float().sum().item() for k in ks}


def count_params(model):
    return sum(p.numel() for p in model.parameters())


@torch.no_grad()
def evaluate(model, loader, max_batches=None):
    """Top-1 / Top-5 accuracy over a loader."""
    model.eval()
    top1, top5, n = 0.0, 0.0, 0
    for i, (x, y) in enumerate(loader):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out = model(x)
        c = topk_correct(out, y, ks=(1, 5))
        top1 += c[1]; top5 += c[5]; n += y.size(0)
        if max_batches is not None and (i + 1) >= max_batches:
            break
    return {"top1": top1 / n, "top5": top5 / n}


def save_checkpoint(model, path, extra=None):
    payload = {"model": model.state_dict()}
    if extra:
        payload.update(extra)
    torch.save(payload, path)


def load_checkpoint(model, path):
    payload = torch.load(path, map_location=device)
    model.load_state_dict(payload["model"])
    return model


def make_scheduler(optimizer, warmup_epochs, total_epochs):
    """Linear warmup then cosine decay (per-epoch granularity)."""
    def lr_lambda(epoch):
        if warmup_epochs > 0 and epoch < warmup_epochs:
            return float(epoch + 1) / float(warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


## 4. Teacher — ResNet50 (pretrained, frozen)

ResNet50 is already trained on ImageNet-1k, so there is nothing to fine-tune. We
load the stronger `IMAGENET1K_V2` weights, freeze the model, and use it purely as
a fixed source of soft logits and intermediate feature maps.


In [ ]:
def build_resnet50_teacher():
    weights = getattr(models.ResNet50_Weights, "IMAGENET1K_V2",
                      models.ResNet50_Weights.IMAGENET1K_V1)
    model = models.resnet50(weights=weights)
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
    return model.to(device)


teacher = build_resnet50_teacher()
print("ResNet50 teacher params:", f"{count_params(teacher):,}")

# Reference accuracy of the teacher on the validation set (E1).
_eval_batches = 50 if CFG.quick_run else None
res_e1 = evaluate(teacher, val_loader, max_batches=_eval_batches)
print(f"E1 ResNet50 teacher | val top1 {res_e1['top1']*100:.2f}% | "
      f"val top5 {res_e1['top5']*100:.2f}%")


## 5. Student — MobileNetV2

In [ ]:
def build_mobilenetv2(num_classes=CFG.num_classes, pretrained=CFG.student_pretrained):
    weights = models.MobileNet_V2_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.mobilenet_v2(weights=weights)
    # 1000-way head already matches ImageNet; rebuild only if num_classes differs.
    if num_classes != 1000:
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)
    return model.to(device)


print("MobileNetV2 student params:",
      f"{count_params(build_mobilenetv2(pretrained=False)):,}")


## 6. Distillation losses

* **Response-based (Hinton).** KL divergence between temperature-softened teacher
  and student logits, scaled by `T^2`.
* **Attention Transfer (Zagoruyko & Komodakis).** Match spatial **attention maps**
  (channel-wise squared activations, L2-normalised) between teacher and student at
  several stages. Channel counts may differ; only spatial sizes must align.


In [ ]:
def hinton_kd_loss(student_logits, teacher_logits, T):
    """Response-based KD: KL(softmax(t/T) || softmax(s/T)) * T^2."""
    s_log = F.log_softmax(student_logits / T, dim=1)
    t_prob = F.softmax(teacher_logits / T, dim=1)
    return F.kl_div(s_log, t_prob, reduction="batchmean") * (T * T)


def attention_map(feat):
    """(B, C, H, W) -> (B, H*W) L2-normalised attention vector."""
    am = feat.pow(2).mean(dim=1)            # (B, H, W)
    am = am.view(am.size(0), -1)            # (B, H*W)
    return F.normalize(am, p=2, dim=1)


def attention_transfer_loss(student_feats, teacher_feats):
    """Mean squared difference of normalised attention maps over matched stages."""
    loss = 0.0
    for s, t in zip(student_feats, teacher_feats):
        loss = loss + (attention_map(s) - attention_map(t.detach())).pow(2).mean()
    return loss


In [ ]:
class FeatureCollector:
    """Capture outputs of selected modules (in registration order) per forward."""
    def __init__(self, modules):
        self.features = []
        self._handles = [m.register_forward_hook(self._hook) for m in modules]

    def _hook(self, module, inp, out):
        self.features.append(out)

    def clear(self):
        self.features = []

    def remove(self):
        for h in self._handles:
            h.remove()


def teacher_at_layers(model):
    # ResNet50 stages -> spatial 28x28, 14x14, 7x7 at 224 input.
    return [model.layer2, model.layer3, model.layer4]


def student_at_layers(model):
    # MobileNetV2 blocks with matching spatial sizes: 28x28, 14x14, 7x7.
    return [model.features[6], model.features[13], model.features[18]]


## 7. Unified training loop (baseline + all KD variants)

One function trains the student under any objective. The **recipe is identical**
across runs; the flags `use_kd` / `use_at` select the experiment:

* Baseline (E2): `use_kd=False, use_at=False`
* Response KD (E3): `use_kd=True, use_at=False`
* Attention Transfer (E4): `use_kd=False, use_at=True`
* Response + AT (E5): `use_kd=True, use_at=True`


In [ ]:
def train_student(tag, ckpt_path, teacher=None, use_kd=False, use_at=False,
                  temperature=KD["temperature"], kd_alpha=KD["kd_alpha"],
                  at_weight=KD["at_weight"]):
    """Train MobileNetV2 with the shared recipe; KD terms toggled by flags."""
    epochs = QUICK["epochs"] if CFG.quick_run else RECIPE["epochs"]
    max_eval = QUICK["max_eval_batches"] if CFG.quick_run else None

    set_seed(CFG.seed)
    student = build_mobilenetv2()

    s_col = t_col = None
    if use_at:
        s_col = FeatureCollector(student_at_layers(student))
        t_col = FeatureCollector(teacher_at_layers(teacher))

    if use_kd or use_at:
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad = False

    criterion = nn.CrossEntropyLoss(label_smoothing=RECIPE["label_smoothing"])
    optimizer = torch.optim.SGD(student.parameters(), lr=RECIPE["lr"],
                                momentum=RECIPE["momentum"],
                                weight_decay=RECIPE["weight_decay"],
                                nesterov=RECIPE["nesterov"])
    scheduler = make_scheduler(optimizer, RECIPE["warmup_epochs"], epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

    history = {"train_loss": [], "train_top1": [], "val_top1": [], "val_top5": []}
    best_val, best_epoch = -1.0, -1

    print("=" * 80)
    print(f"[{tag}] use_kd={use_kd}, use_at={use_at} | epochs={epochs} | "
          f"lr={RECIPE['lr']}, wd={RECIPE['weight_decay']}, bs(quick={CFG.quick_run})")
    if use_kd:
        print(f"  KD: T={temperature}, kd_alpha={kd_alpha} (CE weight={1-kd_alpha})")
    if use_at:
        print(f"  AT: at_weight={at_weight}")
    print("=" * 80)

    try:
        for epoch in range(epochs):
            student.train()
            running_loss, correct1, n = 0.0, 0.0, 0
            lr_now = optimizer.param_groups[0]["lr"]
            for x, y in train_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                if use_at:
                    s_col.clear(); t_col.clear()
                optimizer.zero_grad(set_to_none=True)

                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    if use_kd or use_at:
                        with torch.no_grad():
                            t_logits = teacher(x)
                    s_logits = student(x)

                    ce = criterion(s_logits, y)
                    loss = ce
                    if use_kd:
                        kd = hinton_kd_loss(s_logits, t_logits, temperature)
                        loss = (1.0 - kd_alpha) * ce + kd_alpha * kd
                    if use_at:
                        at = attention_transfer_loss(s_col.features, t_col.features)
                        loss = loss + at_weight * at

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                running_loss += loss.item() * y.size(0)
                correct1 += topk_correct(s_logits, y, ks=(1,))[1]
                n += y.size(0)
            scheduler.step()

            train_loss = running_loss / n
            train_top1 = correct1 / n
            val = evaluate(student, val_loader, max_batches=max_eval)
            gap = train_top1 - val["top1"]
            history["train_loss"].append(train_loss)
            history["train_top1"].append(train_top1)
            history["val_top1"].append(val["top1"])
            history["val_top5"].append(val["top5"])

            if val["top1"] > best_val:
                best_val, best_epoch = val["top1"], epoch + 1
                save_checkpoint(student, ckpt_path,
                                extra={"epoch": epoch, "val_top1": best_val})
                star = "  <-- best"
            else:
                star = ""

            print(f"[{tag}] epoch {epoch+1:02d}/{epochs} | lr {lr_now:.4f} | "
                  f"train_loss {train_loss:.3f} | train_top1 {train_top1*100:.2f}% | "
                  f"val_top1 {val['top1']*100:.2f}% | val_top5 {val['top5']*100:.2f}% | "
                  f"gap {gap*100:+.2f}%{star}")
    finally:
        if s_col is not None: s_col.remove()
        if t_col is not None: t_col.remove()

    print(f"[{tag}] best val top1: {best_val*100:.2f}% @ epoch {best_epoch}")
    history["best_epoch"] = best_epoch
    history["best_val"] = best_val
    return history


## 8. E2 — MobileNetV2 baseline (CrossEntropy only)

In [ ]:
E2_CKPT = os.path.join(CFG.ckpt_dir, "e2_mobilenetv2_baseline.pth")
hist_e2 = train_student("E2-Baseline", E2_CKPT, teacher=None,
                        use_kd=False, use_at=False)
res_e2 = evaluate(load_checkpoint(build_mobilenetv2(), E2_CKPT), val_loader,
                  max_batches=(QUICK["max_eval_batches"] if CFG.quick_run else None))
print(f"E2 baseline | val top1 {res_e2['top1']*100:.2f}% | top5 {res_e2['top5']*100:.2f}%")


## 9. E3 — Response-based KD (Hinton)

In [ ]:
E3_CKPT = os.path.join(CFG.ckpt_dir, "e3_mobilenetv2_response_kd.pth")
hist_e3 = train_student("E3-ResponseKD", E3_CKPT, teacher=teacher,
                        use_kd=True, use_at=False)
res_e3 = evaluate(load_checkpoint(build_mobilenetv2(), E3_CKPT), val_loader,
                  max_batches=(QUICK["max_eval_batches"] if CFG.quick_run else None))
print(f"E3 response KD | val top1 {res_e3['top1']*100:.2f}% | top5 {res_e3['top5']*100:.2f}%")


## 10. E4 — Attention Transfer

In [ ]:
E4_CKPT = os.path.join(CFG.ckpt_dir, "e4_mobilenetv2_attention.pth")
hist_e4 = train_student("E4-AttentionTransfer", E4_CKPT, teacher=teacher,
                        use_kd=False, use_at=True)
res_e4 = evaluate(load_checkpoint(build_mobilenetv2(), E4_CKPT), val_loader,
                  max_batches=(QUICK["max_eval_batches"] if CFG.quick_run else None))
print(f"E4 attention transfer | val top1 {res_e4['top1']*100:.2f}% | top5 {res_e4['top5']*100:.2f}%")


## 11. E5 — Response KD + Attention Transfer

In [ ]:
E5_CKPT = os.path.join(CFG.ckpt_dir, "e5_mobilenetv2_response_attention.pth")
hist_e5 = train_student("E5-Response+AT", E5_CKPT, teacher=teacher,
                        use_kd=True, use_at=True)
res_e5 = evaluate(load_checkpoint(build_mobilenetv2(), E5_CKPT), val_loader,
                  max_batches=(QUICK["max_eval_batches"] if CFG.quick_run else None))
print(f"E5 response+AT | val top1 {res_e5['top1']*100:.2f}% | top5 {res_e5['top5']*100:.2f}%")


## 12. Results comparison

In [ ]:
rows = [
    ["E1", "ResNet50",    "Teacher (pretrained)",          res_e1, None],
    ["E2", "MobileNetV2", "Baseline (CE)",                 res_e2, hist_e2],
    ["E3", "MobileNetV2", "Response KD",                   res_e3, hist_e3],
    ["E4", "MobileNetV2", "Attention Transfer",            res_e4, hist_e4],
    ["E5", "MobileNetV2", "Response KD + AT",              res_e5, hist_e5],
]

def gap_of(h):
    if h is None:
        return None
    i = int(np.argmax(h["val_top1"]))
    return round((h["train_top1"][i] - h["val_top1"][i]) * 100, 2)

comparison = pd.DataFrame([{
    "Exp": r[0],
    "Model": r[1],
    "Method": r[2],
    "Val Top-1 (%)": round(r[3]["top1"] * 100, 2),
    "Val Top-5 (%)": round(r[3]["top5"] * 100, 2),
    "Best Val (%)": (round(r[4]["best_val"] * 100, 2) if r[4] else None),
    "Train-Val Gap (%)": gap_of(r[4]),
} for r in rows])

base = comparison.loc[comparison["Exp"] == "E2", "Val Top-1 (%)"].iloc[0]
comparison["vs Baseline (pp)"] = (comparison["Val Top-1 (%)"] - base).round(2)
print(comparison.to_string(index=False))
comparison


In [ ]:
# ----- Validation Top-1 curves (students only) -----
plt.figure(figsize=(8, 5))
for name, h in [("E2 Baseline", hist_e2), ("E3 Response KD", hist_e3),
                ("E4 Attention", hist_e4), ("E5 Response+AT", hist_e5)]:
    plt.plot([v * 100 for v in h["val_top1"]], marker="o", label=name)
plt.axhline(res_e1["top1"] * 100, ls="--", color="gray", label="ResNet50 teacher")
plt.title("MobileNetV2 student — validation Top-1 by distillation method")
plt.xlabel("epoch"); plt.ylabel("val top-1 (%)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [ ]:
# ----- Final test (val) Top-1 bar chart -----
plt.figure(figsize=(8, 5))
labels = comparison["Exp"] + "\n" + comparison["Method"]
bars = plt.bar(labels, comparison["Val Top-1 (%)"], color="steelblue")
for b, v in zip(bars, comparison["Val Top-1 (%)"]):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.3, f"{v:.1f}",
             ha="center", va="bottom", fontsize=9)
plt.title("ImageNet-1k Top-1 by experiment"); plt.ylabel("val top-1 (%)")
plt.xticks(rotation=15); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()


## 13. Conclusion & how to scale up

* **Fixed recipe, varying KD.** Baseline (E2) and the three distillation variants
  (E3 response, E4 attention transfer, E5 response+AT) all share the **same**
  MobileNetV2 recipe — the comparison isolates the effect of the distillation
  method. Tune only the `KD` dict (`temperature`, `kd_alpha`, `at_weight`).

* **Adjusting distillation methods.**
  * *Response KD:* try `temperature` in {2, 4, 6} and `kd_alpha` in {0.5 … 0.95}.
  * *Attention Transfer:* `at_weight` (often 100–1000) and which stages to match.
  * *Response + AT:* keep both, balancing `kd_alpha` and `at_weight`.

* **Full run.** Set `CFG.quick_run = False`, point `CFG.data_root` at a real
  ImageNet-1k folder (ImageFolder `train/` + `val/`), use a multi-GPU machine, and
  the standard `RECIPE` (SGD, cosine, label smoothing, ~100+ epochs) reaches the
  expected MobileNetV2 accuracy without overfitting.
